In [5]:
import ollama

def run_agent(user_message):
    response = ollama.chat(
        model="llama3.2",
        messages=[{"role": "user", "content": user_message}]
    )
    return response["message"]["content"]

print(run_agent("What are common diabetes risk factors?"))

There are several common diabetes risk factors that can increase your likelihood of developing type 2 diabetes. Some of the most significant risk factors include:

1. **Family History**: Having a first-degree relative (parent or sibling) with diabetes increases your risk.
2. **Obesity**: Being overweight or obese, particularly around the waistline, is a major risk factor for developing type 2 diabetes.
3. **Physical Inactivity**: A sedentary lifestyle can increase your risk of developing insulin resistance and type 2 diabetes.
4. **Age**: The risk of developing type 2 diabetes increases with age, especially after 45 years old.
5. **Ethnicity**: Certain ethnic groups, such as African Americans, Hispanics/Latinos, American Indians, and Asian Americans, are at higher risk for developing type 2 diabetes.
6. **History of Gestational Diabetes**: Women who have had gestational diabetes during pregnancy are at increased risk of developing type 2 diabetes later in life.
7. **Polycystic Ovary Sy

In [6]:
import ollama
import json

# --- Tool 1: Symptom → Disease guesser ---
def guess_disease(symptoms: str) -> str:
    response = ollama.chat(
        model="llama3.2",
        messages=[{
            "role": "user",
            "content": f"""You are a medical AI assistant.
Given these symptoms: {symptoms}
List the top 3 most likely conditions, each with:
- Condition name
- Likelihood (high/medium/low)
- Key matching symptoms
- Recommended next step
Format as JSON. Be concise."""
        }]
    )
    return response["message"]["content"]

# --- Tool 2: Drug interaction checker ---
def check_drug_interactions(drugs: str) -> str:
    response = ollama.chat(
        model="llama3.2",
        messages=[{
            "role": "user",
            "content": f"""Check interactions between: {drugs}
For each pair, state: severity (mild/moderate/severe), effect, and advice.
Format as JSON."""
        }]
    )
    return response["message"]["content"]

# --- Tool 3: Vitals risk assessor ---
def assess_vitals(vitals: str) -> str:
    response = ollama.chat(
        model="llama3.2",
        messages=[{
            "role": "user",
            "content": f"""Assess these patient vitals: {vitals}
Flag any abnormal values, their severity, and what they may indicate.
Format as JSON with keys: vital, value, status (normal/warning/critical), note."""
        }]
    )
    return response["message"]["content"]

# --- Tool 4: Triage urgency classifier ---
def triage_patient(description: str) -> str:
    response = ollama.chat(
        model="llama3.2",
        messages=[{
            "role": "user",
            "content": f"""Triage this patient case: {description}
Classify urgency as: immediate / urgent / semi-urgent / non-urgent.
Give reasoning and recommended care setting (ER, GP, telehealth, home care).
Format as JSON."""
        }]
    )
    return response["message"]["content"]

# Tool registry
TOOLS = {
    "guess_disease": guess_disease,
    "check_drug_interactions": check_drug_interactions,
    "assess_vitals": assess_vitals,
    "triage_patient": triage_patient,
}

print("✓ Tools loaded:", list(TOOLS.keys()))

✓ Tools loaded: ['guess_disease', 'check_drug_interactions', 'assess_vitals', 'triage_patient']


In [7]:
def health_agent(patient_query: str):
    """Main agent that routes patient queries to the right tool."""

    # Step 1: Ask the LLM which tool to use
    router_response = ollama.chat(
        model="llama3.2",
        messages=[{
            "role": "system",
            "content": """You are a healthtech agent router.
Available tools:
- guess_disease: use when patient describes symptoms
- check_drug_interactions: use when medications are mentioned
- assess_vitals: use when numbers like BP, HR, temp are given
- triage_patient: use when urgency/severity needs classifying

Reply ONLY with JSON: {"tool": "tool_name", "input": "extracted input"}"""
        }, {
            "role": "user",
            "content": patient_query
        }]
    )

    raw = router_response["message"]["content"]

    try:
        clean = raw.replace("```json", "").replace("```", "").strip()
        routing = json.loads(clean)
        tool_name = routing["tool"]
        tool_input = routing["input"]
    except:
        print("Router error — defaulting to guess_disease")
        tool_name = "guess_disease"
        tool_input = patient_query

    print(f"\n🔧 Using tool: {tool_name}")
    print(f"📥 Input: {tool_input}\n")

    # Step 2: Run the tool
    result = TOOLS[tool_name](tool_input)

    # Step 3: Summarise for the user in plain English
    summary = ollama.chat(
        model="llama3.2",
        messages=[{
            "role": "user",
            "content": f"""Summarise this medical analysis in 3-4 clear sentences
for a non-medical audience. Always end with a disclaimer that
this is AI-generated and not a substitute for professional advice.

Analysis: {result}"""
        }]
    )

    print("📋 Result:\n", result)
    print("\n💬 Summary:\n", summary["message"]["content"])

In [9]:
health_agent("I was feeling dizzy and nauseous and light-headed over the weekend after a sudden pilates workout. I tend to have low blood pressure. I was sleep deprived and dehydrated. what was going on with me?")


🔧 Using tool: guess_disease
📥 Input: {"symptoms": ["dizziness", "nausea", "light-headedness"], "exposure": ["pilates workout"]}

📋 Result:
 Based on the provided symptoms and exposure to a Pilates workout, here are the top 3 most likely conditions:

```json
{
    "conditions": [
        {
            "name": "Vestibular Migraine",
            "likelihood": "high",
            "key_symptoms": ["dizziness", "nausea"],
            "next_step": "Seek medical attention and consider an Epley maneuver for acute vertigo"
        },
        {
            "name": "Dehydration",
            "likelihood": "medium",
            "key_symptoms": ["light-headedness"],
            "next_step": "Drink plenty of water and monitor electrolyte levels"
        },
        {
            "name": "Overexertion",
            "likelihood": "low",
            "key_symptoms": ["dizziness", "nausea"],
            "next_step": "Rest and recover; consider adjusting exercise intensity or frequency"
        }
    ]
}
`

In [8]:
# Run all 4 tools with realistic patient cases

# 1. Symptom checker
health_agent("Patient is a 35-year-old male with persistent cough for 3 weeks, night sweats, unexplained weight loss of 5kg, and mild fever")

print("\n" + "="*60 + "\n")

# 2. Drug interaction
health_agent("My patient is currently taking metformin and lisinopril, and I want to add aspirin to their regimen")

print("\n" + "="*60 + "\n")

# 3. Vitals assessment
health_agent("Vitals: BP 145/95, heart rate 112 bpm, temperature 38.8°C, SpO2 91%, respiratory rate 24")

print("\n" + "="*60 + "\n")

# 4. Triage
health_agent("52-year-old woman, sudden severe headache she describes as the worst of her life, stiff neck, sensitive to light")


🔧 Using tool: guess_disease
📥 Input: {"age": 35, "sex": "male", "symptoms": ["persistent cough", "night sweats", "unexplained weight loss", "mild fever"]}

📋 Result:
 ```json
{
  "conditions": [
    {
      "name": "Lung Cancer",
      "likelihood": "high",
      "matchingSymptoms": ["persistent cough", "unexplained weight loss"],
      "nextStep": "Imaging studies (e.g., CT scan) and potentially a biopsy"
    },
    {
      "name": "Pulmonary Tuberculosis",
      "likelihood": "medium",
      "matchingSymptoms": ["persistent cough", "night sweats"],
      "nextStep": "TB testing (e.g., sputum analysis or chest X-ray)"
    },
    {
      "name": "Chronic Bronchitis",
      "likelihood": "low",
      "matchingSymptoms": ["persistent cough"],
      "nextStep": "Review of smoking history and potential management with medication"
    }
  ]
}
```

Note: These conditions are not definitive diagnoses, but rather based on the provided symptoms. A proper diagnosis can only be made by a qualifi

In [2]:
import sys
!{sys.executable} -m pip install requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 1.3 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.2/315.2 kB 5.0 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip available: 22.2.2 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip


In [3]:
import requests
import pandas as pd

# Fetch common symptom-condition mappings from OpenFDA
url = "https://api.fda.gov/drug/event.json?count=patient.reaction.reactionmeddrapt.exact&limit=20"
response = requests.get(url)
data = response.json()

# Turn into a dataframe
symptoms_df = pd.DataFrame(data["results"])
symptoms_df.columns = ["symptom", "count"]
print(symptoms_df.head(10))

            symptom    count
0  DRUG INEFFECTIVE  1279723
1             DEATH   837445
2     OFF LABEL USE   835572
3            NAUSEA   763803
4           FATIGUE   753190
5         DIARRHOEA   621404
6          HEADACHE   607907
7              PAIN   599257
8          DYSPNOEA   549489
9         DIZZINESS   483156


In [5]:
data['results']

[{'term': 'DRUG INEFFECTIVE', 'count': 1279723},
 {'term': 'DEATH', 'count': 837445},
 {'term': 'OFF LABEL USE', 'count': 835572},
 {'term': 'NAUSEA', 'count': 763803},
 {'term': 'FATIGUE', 'count': 753190},
 {'term': 'DIARRHOEA', 'count': 621404},
 {'term': 'HEADACHE', 'count': 607907},
 {'term': 'PAIN', 'count': 599257},
 {'term': 'DYSPNOEA', 'count': 549489},
 {'term': 'DIZZINESS', 'count': 483156},
 {'term': 'VOMITING', 'count': 453902},
 {'term': 'MALAISE', 'count': 428234},
 {'term': 'RASH', 'count': 419095},
 {'term': 'ARTHRALGIA', 'count': 400855},
 {'term': 'ASTHENIA', 'count': 365401},
 {'term': 'PRURITUS', 'count': 361429},
 {'term': 'PYREXIA', 'count': 336948},
 {'term': 'FALL', 'count': 322939},
 {'term': 'PNEUMONIA', 'count': 313222},
 {'term': 'CONDITION AGGRAVATED', 'count': 291665}]

In [6]:
!pip install streamlit ollama requests pandas

     |████████████████████████████████| 10.1 MB 1.5 MB/s eta 0:00:01
     |████████████████████████████████| 32.7 MB 1.2 MB/s eta 0:00:011
     |████████████████████████████████| 212 kB 104.4 MB/s eta 0:00:01
     |████████████████████████████████| 427 kB 7.2 MB/s eta 0:00:01
     |████████████████████████████████| 11.3 MB 8.0 MB/s eta 0:00:01
     |████████████████████████████████| 731 kB 82.7 MB/s eta 0:00:01
     |████████████████████████████████| 451 kB 10.0 MB/s eta 0:00:01
     |████████████████████████████████| 62 kB 7.6 MB/s  eta 0:00:01
  Attempting uninstall: tenacity
    Found existing installation: tenacity 8.0.1
    Uninstalling tenacity-8.0.1:
      Successfully uninstalled tenacity-8.0.1
  Attempting uninstall: protobuf
    Found existing installation: protobuf 3.19.1
    Uninstalling protobuf-3.19.1:
      Successfully uninstalled protobuf-3.19.1
